# Title — Approach

**Competition:** Title
**Approach:**

| | Score |
|---|---|
| **CV (OOF AUC)** |  |
| **Public LB** |  |

## 1. Environment


In [2]:
# Standard library
import os
import json
import random
import subprocess
import warnings
import zipfile
from pathlib import Path

# Numerical / data
import numpy as np
import pandas as pd

# Machine learning / experiment tracking
import lightgbm as lgb
import torch
import wandb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.model_selection import GroupKFold
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')


In [ ]:
# Global config
global_params = dict(
    comp_slug       = "",
    target_col      = "",
    id_col          = "id",
    n_folds         = 5,
    random_state    = 42,
    kaggle_comp_dir = Path(""),
    kaggle_ext_path = Path(""),
    local_data_dir  = Path("data/raw"),
)

# Set all random seeds
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(global_params['random_state'])
np.random.seed(global_params['random_state'])
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(global_params['random_state'])
    torch.set_float32_matmul_precision('high')

global_params['local_data_dir'].mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")


Device: cuda


## 2. Credentials
Do not modify any cells in Section 2 (Credentials)

In [ ]:
class NotebookEnv:
    def __init__(self) -> None:
        self.name: str = self._detect()
        print(f"Running environment: {self.name}")

    @staticmethod
    def _detect() -> str:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
          return "kaggle"

        if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_BACKEND_VERSION"):
            return "colab"

        # Fallbacks
        if "google.colab" in globals():
            return "colab"

        # Files that tend to exist in Colab images
        if Path("/colab_requirements.txt").exists():
            return "colab"

        # Files/dirs that tend to exist in Kaggle
        if Path("/kaggle/input").exists() or Path("/kaggle/working").exists():
            return "kaggle"

        return "local"

    def get_secret(self, key: str) -> str:
        if self.name == "kaggle":
            from kaggle_secrets import UserSecretsClient  # type: ignore
            value = UserSecretsClient().get_secret(key)
        elif self.name == "colab":
            from google.colab import userdata  # type: ignore
            value = userdata.get(key)
        else:
            value = os.environ.get(key)
        if not value:
            raise EnvironmentError(
                f"✅ Environment variable '{key}' is not set."
                " Please set it in ~/.bashrc or similar."
            )
        return value

    def setup_kaggle(self) -> None:
        dst_dir = Path.home() / ".kaggle"
        dst_dir.mkdir(parents=True, exist_ok=True)
        cred = {
            "username": self.get_secret("KAGGLE_USERNAME"),
            "key":      self.get_secret("KAGGLE_KEY"),
        }
        dst = dst_dir / "kaggle.json"
        dst.write_text(json.dumps(cred))
        try:
            dst.chmod(0o600)
        except Exception:
            pass
        print(f"✅ Kaggle authentication completed (user: {cred['username']}, env: {self.name})")

    def setup_wandb(self) -> None:
        wandb.login(key=self.get_secret("WANDB_API_KEY"), relogin=True)
        print(f"wandb login completed (env: {self.name})")

    def download_competition(self, data_dir: Path) -> None:
        data_dir.mkdir(parents=True, exist_ok=True)
        self._run(["kaggle", "config", "view"])
        self._run([
            "kaggle", "competitions", "download",
            "-c", global_params['comp_slug'],
            "-p", str(data_dir),
            "--force",
        ])
        zips = list(data_dir.glob("*.zip"))
        if not zips:
            raise FileNotFoundError(f"No zip found in {data_dir} after download.")
        for zp in zips:
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(data_dir)
            print("Unzipped:", zp.name)

    @staticmethod
    def _run(cmd: list[str]) -> None:
        subprocess.run(cmd, capture_output=True, text=True, check=True)

env = NotebookEnv()


Running environment: colab


In [5]:
# Setup kaggle datasets if not on kaggle env
if env.name != "kaggle":
    kaggle_env = env.setup_kaggle()

wandb_env = env.setup_wandb()


✅ Kaggle 認証完了  (user: bloodymonday, env: colab)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jojoto8845 (jojoto8845-line) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ wandb login 完了 (env: colab)


In [ ]:
if wandb.run is not None:
    wandb.finish()

wandb_config = {
    'competition':    global_params['comp_slug'],
    'environment':    env.name,
    'n_folds':        global_params['n_folds'],
    'random_state':   global_params['random_state'],
}

# Fill in your wandb entity, project, and job_type
run = wandb.init(
    entity="",
    project="",
    job_type="",
    config=wandb_config,
)


## 3. Data Loading


In [ ]:
kaggle_dir = global_params['kaggle_comp_dir']
local_dir  = global_params['local_data_dir']

if kaggle_dir.exists():
    data_dir = kaggle_dir
    print("[Kaggle] Using mounted competition data:", data_dir)
else:
    data_dir = local_dir
    env.download_competition(local_dir)
    print("✅ Download completed. Using local data:", data_dir)


[Download] Kaggle API でデータを取得します...
Unzipped: march-machine-learning-mania-2026.zip
✅ ダウンロード完了
